Importing the Dependencies

In [34]:
from ultralytics import YOLO
import torch
import pandas as pd
from pathlib import Path
import numpy as np
import os

In [2]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: NVIDIA H100 NVL


In [24]:
DATA_YAML = "/data/UG/Kiranmoy/datasets/MHCD2022_YOLO/data.yaml"

In [25]:
model = YOLO("yolo11s.pt")

In [7]:
results = model.train(
    data=DATA_YAML,

    epochs=100,

    imgsz=640,

    batch=64,

    workers=16,

    device=0,

    cache=True,

    amp=True,

    seed=42,

    patience=30,

    project="MHCD_Benchmarks",

    name="YOLO11s_Baseline",
    exist_ok=True

)

Ultralytics 8.4.60 🚀 Python-3.10.20 torch-2.0.1+cu118 CUDA:0 (NVIDIA H100 NVL, 95235MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/data/UG/Kiranmoy/datasets/MHCD2022_YOLO/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLO11s_Baseline, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto

In [8]:
print(results.save_dir)

/data/UG/Kiranmoy/Repo_Destination/High-Level-Military-Camouflage-Detection/runs/detect/MHCD_Benchmarks/YOLO11s_Baseline


In [10]:
best_model = YOLO(
    str(
        Path(results.save_dir)
        / "weights"
        / "best.pt"
    )
)

In [11]:
metrics = best_model.val(
    data="/data/UG/Kiranmoy/datasets/MHCD2022_YOLO/data.yaml",
    split="test",
    device=0
)

Ultralytics 8.4.60 🚀 Python-3.10.20 torch-2.0.1+cu118 CUDA:0 (NVIDIA H100 NVL, 95235MiB)
YOLO11s summary (fused): 101 layers, 9,414,735 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7.8±4.5 MB/s, size: 117.9 KB)
val: Scanning /data/UG/Kiranmoy/datasets/MHCD2022_YOLO/labels/test.cache... 600 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 600/600 34.5Mit/s 0.0s
val: /data/UG/Kiranmoy/datasets/MHCD2022_YOLO/images/test/000224.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 5.1it/s 7.4s0.1ss
                   all        600        920      0.803      0.582       0.64      0.407
                person        531        718      0.926      0.763      0.849      0.537
      military vehicle         35         53      0.594      0.358      0.355      0.231
                  tank         49         89      0.857      0.604      0.679      0.447
  

In [16]:
ap50 = metrics.box.map50
ap75 = metrics.box.map75
ap5095 = metrics.box.map

precision = metrics.box.mp
recall = metrics.box.mr

f1 = (
    2 * precision * recall
) / (
    precision + recall + 1e-8
)

print(f"AP50      : {ap50:.4f}")
print(f"AP75      : {ap75:.4f}")
print(f"AP50-95   : {ap5095:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1        : {f1:.4f}")

AP50      : 0.6400
AP75      : 0.4303
AP50-95   : 0.4069
Precision : 0.8030
Recall    : 0.5817
F1        : 0.6747


In [17]:
metrics.box.maps

array([    0.53718,     0.23058,     0.44705,     0.37124,     0.44857])

In [22]:
benchmark = pd.DataFrame([
    {
        "Model": "YOLO11-s",
        "AP50": ap50,
        "AP75": ap75,
        "AP50-95": ap5095,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }
])

benchmark

,Model,AP50,AP75,AP50-95,Precision,Recall,F1
0,YOLO11-s,0.639971,0.430297,0.406923,0.803039,0.581678,0.674665


In [26]:
benchmark.to_csv(
    "YOLO11s_Benchmark.csv",
    index=False
)

In [ ]:
save_dir = Path(
    "/data/UG/Kiranmoy/Repo_Destination/High-Level-Military-Camouflage-Detection/runs/detect/MHCD_Benchmarks/YOLO11s_Baseline"
)

for f in save_dir.iterdir():
    print(f.name)

val_batch1_pred.jpg
train_batch3421.jpg
weights
results.csv
labels.jpg
train_batch3420.jpg
val_batch2_pred.jpg
BoxPR_curve.png
val_batch2_labels.jpg
confusion_matrix.png
train_batch3422.jpg
train_batch1.jpg
args.yaml
val_batch1_labels.jpg
train_batch2.jpg
train_batch0.jpg
confusion_matrix_normalized.png
BoxP_curve.png
BoxR_curve.png
val_batch0_labels.jpg
BoxF1_curve.png
results.png
val_batch0_pred.jpg


In [28]:
best_model.predict(
    source="/data/UG/Kiranmoy/datasets/MHCD2022_YOLO/images/test",
    save=True,
    conf=0.25,
    device=0
)


image 1/600 /data/UG/Kiranmoy/datasets/MHCD2022_YOLO/images/test/000013.jpg: 416x640 4 persons, 226.7ms
image 2/600 /data/UG/Kiranmoy/datasets/MHCD2022_YOLO/images/test/000022.jpg: 448x640 2 persons, 1 military vehicle, 179.7ms
image 3/600 /data/UG/Kiranmoy/datasets/MHCD2022_YOLO/images/test/000025.jpg: 544x640 4 persons, 1 warship, 167.4ms
image 4/600 /data/UG/Kiranmoy/datasets/MHCD2022_YOLO/images/test/000028.jpg: 512x640 1 person, 164.1ms
image 5/600 /data/UG/Kiranmoy/datasets/MHCD2022_YOLO/images/test/000033.jpg: 416x640 6 persons, 1 tank, 7.1ms
image 6/600 /data/UG/Kiranmoy/datasets/MHCD2022_YOLO/images/test/000034.jpg: 448x640 2 military vehicles, 7.1ms
image 7/600 /data/UG/Kiranmoy/datasets/MHCD2022_YOLO/images/test/000038.jpg: 448x640 (no detections), 6.4ms
image 8/600 /data/UG/Kiranmoy/datasets/MHCD2022_YOLO/images/test/000043.jpg: 448x640 1 person, 1 military vehicle, 9.7ms
image 9/600 /data/UG/Kiranmoy/datasets/MHCD2022_YOLO/images/test/000054.jpg: 480x640 1 military vehicl

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'person', 1: 'military vehicle', 2: 'tank', 3: 'aeroplane', 4: 'warship'}
 obb: None
 orig_img: array([[[246, 242, 241],
         [246, 242, 241],
         [246, 242, 241],
         ...,
         [231, 228, 224],
         [231, 228, 224],
         [233, 228, 224]],
 
        [[246, 242, 241],
         [246, 242, 241],
         [246, 242, 241],
         ...,
         [231, 228, 224],
         [231, 228, 224],
         [233, 228, 224]],
 
        [[246, 242, 241],
         [246, 242, 241],
         [246, 242, 241],
         ...,
         [231, 228, 224],
         [231, 228, 224],
         [233, 228, 224]],
 
        ...,
 
        [[ 33,  56,  75],
         [ 41,  64,  83],
         [ 45,  68,  87],
         ...,
         [ 17,  34,  45],
         [ 16,  32,  43],
         [ 16,  31,  42]],
 
        [[ 30,  53,  72],
         [ 38,  61, 